<a href="https://colab.research.google.com/github/tendolkarurja/IPD_Finance/blob/main/model_apply.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
df = pd.read_csv('/content/drive/MyDrive/final_audit_df.csv')

In [28]:
df

,Date,Headline,Headline link,company
0,2022-01-03,Reliance becomes promoter in Sterling and Wils...,https://economictimes.indiatimes.com/industry/...,reliance
1,2022-01-04,Sterling and Wilson Renewable rise 2% as Relia...,https://economictimes.indiatimes.com/markets/s...,reliance
2,2022-01-11,Biggest gainers & losers of the day: Voda Idea...,https://economictimes.indiatimes.com/markets/s...,adani
3,2022-01-12,DERC turns down city discoms’ request to relin...,https://economictimes.indiatimes.com/industry/...,ntpc
4,2022-01-13,Reliance signs agreement with Gujarat to inves...,https://economictimes.indiatimes.com/industry/...,reliance
...,...,...,...,...
811,2025-06-02,Honeywell signs pact with NTPC Green to explor...,https://economictimes.indiatimes.com/industry/...,ntpc
812,2025-06-05,JSW Energy commissions 281 MW renewable energy...,https://economictimes.indiatimes.com/industry/...,jsw
813,2025-06-06,"NTPC Group installed capacity stands at 80,265...",https://economictimes.indiatimes.com/industry/...,ntpc
814,2025-06-06,JSW Energy shares in focus on commissioning 28...,https://economictimes.indiatimes.com/markets/s...,jsw


In [29]:
!pip install transformers torch

In [30]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification,pipeline


In [33]:
sentiment_pipe = pipeline("sentiment-analysis", model="ProsusAI/finbert", top_k=None)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
sentiment_pipe

TextClassificationPipeline: {'model': 'BertForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [35]:
def generate_scores(text):
    try:
        results = sentiment_pipe(text[:512])
        if isinstance(results[0], list):
            return {item['label'].lower(): item['score'] for item in results[0]}

        elif isinstance(results[0], dict):
            output = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
            label = results[0]['label'].lower()
            output[label] = results[0]['score']
            return output

    except Exception as e:
        return {'positive': 0.0, 'negative': 0.0, 'neutral': 1.0}

# Now run the apply again
print("Processing headlines with the robust function...")
scores_df = df['Headline'].apply(generate_scores).apply(pd.Series)

Processing headlines with the robust function...


In [36]:
scores_df

,neutral,positive,negative
0,0.901476,0.088610,0.009914
1,0.027760,0.947282,0.024958
2,0.198752,0.639171,0.162076
3,0.800283,0.079145,0.120572
4,0.062623,0.925366,0.012011
...,...,...,...
811,0.105759,0.884972,0.009269
812,0.910477,0.074887,0.014636
813,0.896316,0.094371,0.009313
814,0.204826,0.787782,0.007392


In [37]:
df_withScores = pd.concat([df, scores_df], axis=1)

In [38]:
df_withScores

,Date,Headline,Headline link,company,neutral,positive,negative
0,2022-01-03,Reliance becomes promoter in Sterling and Wils...,https://economictimes.indiatimes.com/industry/...,reliance,0.901476,0.088610,0.009914
1,2022-01-04,Sterling and Wilson Renewable rise 2% as Relia...,https://economictimes.indiatimes.com/markets/s...,reliance,0.027760,0.947282,0.024958
2,2022-01-11,Biggest gainers & losers of the day: Voda Idea...,https://economictimes.indiatimes.com/markets/s...,adani,0.198752,0.639171,0.162076
3,2022-01-12,DERC turns down city discoms’ request to relin...,https://economictimes.indiatimes.com/industry/...,ntpc,0.800283,0.079145,0.120572
4,2022-01-13,Reliance signs agreement with Gujarat to inves...,https://economictimes.indiatimes.com/industry/...,reliance,0.062623,0.925366,0.012011
...,...,...,...,...,...,...,...
811,2025-06-02,Honeywell signs pact with NTPC Green to explor...,https://economictimes.indiatimes.com/industry/...,ntpc,0.105759,0.884972,0.009269
812,2025-06-05,JSW Energy commissions 281 MW renewable energy...,https://economictimes.indiatimes.com/industry/...,jsw,0.910477,0.074887,0.014636
813,2025-06-06,"NTPC Group installed capacity stands at 80,265...",https://economictimes.indiatimes.com/industry/...,ntpc,0.896316,0.094371,0.009313
814,2025-06-06,JSW Energy shares in focus on commissioning 28...,https://economictimes.indiatimes.com/markets/s...,jsw,0.204826,0.787782,0.007392


In [39]:
# Test with a complex headline that should have mixed sentiment
test_headline = "Adani shares rise despite ongoing regulatory concerns"
raw_output = sentiment_pipe(test_headline)

print("--- RAW MODEL OUTPUT ---")
print(raw_output)

--- RAW MODEL OUTPUT ---
[[{'label': 'positive', 'score': 0.9356290698051453}, {'label': 'negative', 'score': 0.039950791746377945}, {'label': 'neutral', 'score': 0.024420147761702538}]]


In [40]:
df_withScores['cumulative_score'] = df_withScores['positive'] - df_withScores['negative']
df_withScores

,Date,Headline,Headline link,company,neutral,positive,negative,cumulative_score
0,2022-01-03,Reliance becomes promoter in Sterling and Wils...,https://economictimes.indiatimes.com/industry/...,reliance,0.901476,0.088610,0.009914,0.078695
1,2022-01-04,Sterling and Wilson Renewable rise 2% as Relia...,https://economictimes.indiatimes.com/markets/s...,reliance,0.027760,0.947282,0.024958,0.922324
2,2022-01-11,Biggest gainers & losers of the day: Voda Idea...,https://economictimes.indiatimes.com/markets/s...,adani,0.198752,0.639171,0.162076,0.477095
3,2022-01-12,DERC turns down city discoms’ request to relin...,https://economictimes.indiatimes.com/industry/...,ntpc,0.800283,0.079145,0.120572,-0.041427
4,2022-01-13,Reliance signs agreement with Gujarat to inves...,https://economictimes.indiatimes.com/industry/...,reliance,0.062623,0.925366,0.012011,0.913355
...,...,...,...,...,...,...,...,...
811,2025-06-02,Honeywell signs pact with NTPC Green to explor...,https://economictimes.indiatimes.com/industry/...,ntpc,0.105759,0.884972,0.009269,0.875704
812,2025-06-05,JSW Energy commissions 281 MW renewable energy...,https://economictimes.indiatimes.com/industry/...,jsw,0.910477,0.074887,0.014636,0.060251
813,2025-06-06,"NTPC Group installed capacity stands at 80,265...",https://economictimes.indiatimes.com/industry/...,ntpc,0.896316,0.094371,0.009313,0.085058
814,2025-06-06,JSW Energy shares in focus on commissioning 28...,https://economictimes.indiatimes.com/markets/s...,jsw,0.204826,0.787782,0.007392,0.780390


In [42]:
df_withScores.to_csv('finbert_scores_ESG.csv', index= False)